# Covid/Ukr Cross-Graph Eval Results

Reads `eval_results.csv` from this directory and plots inline heatmaps. No figures are saved.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CSV_PATH = Path("eval_results.csv")
DATASET_ORDER = [
    "covid19_twitter",
    "ukr_rus_twitter",
    "midterm",
    "covid_political",
    "election2020",
    "ukr_rus_suspended",
]
TASK_LABELS = {
    "nm": "Neighbor matching",
    "lp": "Temporal link prediction",
    "pl": "Classification",
}
METRICS = ["accuracy", "f1", "roc_auc"]

In [ ]:
df = pd.read_csv(CSV_PATH)
df["shots"] = df["shots"].astype(int)
for col in METRICS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["task_label"] = df["task"].map(TASK_LABELS).fillna(df["task"])
df["dataset"] = pd.Categorical(df["dataset"], categories=DATASET_ORDER, ordered=True)
df = df.sort_values(["split", "task", "dataset", "checkpoint", "shots"])

display(df.head())
display(
    df.groupby(["split", "task", "checkpoint", "shots"], observed=True)
      .size()
      .rename("rows")
      .reset_index()
)

In [ ]:
def plot_metric_heatmap(data, *, split, task, metric):
    subset = data[(data["split"] == split) & (data["task"] == task)].copy()
    if subset.empty or metric not in subset.columns or subset[metric].dropna().empty:
        print(f"skip split={split} task={task} metric={metric}: no data")
        return

    subset["checkpoint_shot"] = (
        subset["checkpoint"].astype(str) + " / " + subset["shots"].astype(str) + " shot"
    )
    col_order = (
        subset[["checkpoint", "shots", "checkpoint_shot"]]
        .drop_duplicates()
        .sort_values(["checkpoint", "shots"])["checkpoint_shot"]
        .tolist()
    )
    pivot = subset.pivot(index="dataset", columns="checkpoint_shot", values=metric)
    pivot = pivot.reindex(DATASET_ORDER).reindex(columns=col_order)

    width = max(7, 1.1 * len(pivot.columns) + 2)
    height = max(3.5, 0.55 * len(pivot.index) + 1.5)
    fig, ax = plt.subplots(figsize=(width, height))
    sns.heatmap(
        pivot,
        ax=ax,
        annot=True,
        fmt=".3f",
        cmap="viridis",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": metric},
    )
    ax.set_title(f"{split} {TASK_LABELS.get(task, task)} - {metric}")
    ax.set_xlabel("checkpoint / shots")
    ax.set_ylabel("dataset")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def plot_all_heatmaps(data, *, split="test", tasks=("nm", "lp", "pl"), metrics=METRICS):
    for task in tasks:
        for metric in metrics:
            plot_metric_heatmap(data, split=split, task=task, metric=metric)

In [ ]:
plot_all_heatmaps(df, split="test")

In [ ]:
# Optional: if the CSV includes validation rows, plot them too.
if "val" in set(df["split"]):
    plot_all_heatmaps(df, split="val")

In [ ]:
# Compact table view for copying into notes or a spreadsheet.
summary_cols = ["split", "dataset", "task", "checkpoint", "shots", "accuracy", "f1", "roc_auc"]
summary = df[summary_cols].sort_values(["split", "task", "dataset", "checkpoint", "shots"])
display(summary)